### Skenario Tugas
Tim Data Engineering e-commerce diminta membangun pipeline ETL produksi yang menyatukan tiga sumber data terdistribusi: transaksi harian (CSV), katalog master produk (JSON Lines), dan rating ulasan pembeli (CSV). Data harus digabungkan tanpa kehilangan data transaksi, ditangani missing values-nya secara presisi, lalu disimpan ke sistem terdistribusi dalam format Parquet terpartisi untuk efisiensi kueri masa depan.

### 1. Menyiapkan Tiga Sumber Data Sintetis
Membangkitkan dataset transaksi (5.000 baris CSV), data master produk (30 item JSON Lines), dan data ulasan pelanggan (3.500 baris CSV) dengan random seed 33 sesuai modul.

In [2]:
import os
import sys
import json
import getpass
import numpy as np
import pandas as pd

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01", "2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000, 50000, 75000, 100000, 150000, 250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — 3500 baris (transaksi tanpa ulasan bernilai null)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print("Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6):,} baris")
print(f"- tugas6_produk.json   : {len(produk):,} produk")
print(f"- tugas6_ulasan.csv    : {len(df_review):,} baris")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5,000 baris
- tugas6_produk.json   : 30 produk
- tugas6_ulasan.csv    : 3,500 baris


### 2. Inisialisasi SparkSession
Mengunci alokasi runtime OpenJDK 11 terisolasi di Arch Linux dan membuat sesi komputasi PySpark.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum as spark_sum, avg, round as spark_round

# Konfigurasi isolasi lingkungan native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Tugas6-ETL-SholahuddinAhmad-250506070") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"User Host : {getpass.getuser()} | Spark Engine : Apache Spark {spark.version}")
print("Status    : SparkSession aktif & terisolasi penuh!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 22:50:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/30 22:50:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


User Host : kaky | Spark Engine : Apache Spark 3.5.9
Status    : SparkSession aktif & terisolasi penuh!


### A. EXTRACT — Membaca Tiga Sumber Data
Mengekstrak berkas CSV transaksi, JSON Lines produk, dan CSV ulasan ke dalam DataFrame PySpark serta memeriksa jumlah baris dan skemanya.

In [5]:
# Ekstraksi ketiga berkas ke dalam Spark DataFrame
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"Data Transaksi : {df_transaksi.count():,} baris")
df_transaksi.printSchema()

print(f"\nData Produk    : {df_produk.count():,} baris")
df_produk.printSchema()

print(f"\nData Ulasan    : {df_ulasan.count():,} baris")
df_ulasan.printSchema()

Data Transaksi : 5,000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)


Data Produk    : 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)


Data Ulasan    : 3,500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



### B. TRANSFORM — Penggabungan Relasional
Menerapkan Left Join ke ulasan agar 1.500 transaksi yang belum diulas tidak terbuang (*zero data loss*) dan Inner Join ke master produk. Kolom `total_pendapatan` ditambahkan dari perkalian unit dan harga.

In [6]:
# Penggabungan DataFrame
df_gabung = df_transaksi \
    .join(df_ulasan, on="order_id", how="left") \
    .join(df_produk, on="product_id", how="inner") \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

print(f"Total baris setelah join: {df_gabung.count():,} (Integritas 100% terjaga)")
df_gabung.select("order_id", "nama_produk", "kategori", "unit_terjual", "harga", "total_pendapatan", "rating").show(5)

Total baris setelah join: 5,000 (Integritas 100% terjaga)
+--------+-----------+------------+------------+------+----------------+------+
|order_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+-----------+------------+------------+------+----------------+------+
|     TX0|  Produk-21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|   Produk-8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|  Produk-25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|   Produk-3|     Fashion|           4| 75000|          300000|     5|
|     TX4|  Produk-19|Rumah Tangga|           6| 75000|          450000|  NULL|
+--------+-----------+------------+------------+------+----------------+------+
only showing top 5 rows



### C. TRANSFORM — Penanganan Data Kosong & Pengayaan
Menambahkan kolom boolean `ada_ulasan` sebelum imputasi, lalu mengisi nilai `rating = null` dengan angka 0 sebagai *sentinel value* (membedakan status belum diulas dari ulasan rating 1).

In [7]:
# 1. Tambahkan flag boolean ada_ulasan sebelum imputasi
df_enriched = df_gabung.withColumn("ada_ulasan", col("rating").isNotNull())

# 2. Imputasi nilai null pada rating dengan angka 0
df_final = df_enriched.na.fill({"rating": 0})

print("Cuplikan data bersih setelah imputasi sentinel 0:")
df_final.select("order_id", "nama_produk", "kategori", "total_pendapatan", "rating", "ada_ulasan").show(5)

Cuplikan data bersih setelah imputasi sentinel 0:
+--------+-----------+------------+----------------+------+----------+
|order_id|nama_produk|    kategori|total_pendapatan|rating|ada_ulasan|
+--------+-----------+------------+----------------+------+----------+
|     TX0|  Produk-21|  Elektronik|           50000|     1|      true|
|     TX1|   Produk-8|     Fashion|         1500000|     0|     false|
|     TX2|  Produk-25|  Elektronik|          100000|     2|      true|
|     TX3|   Produk-3|     Fashion|          300000|     5|      true|
|     TX4|  Produk-19|Rumah Tangga|          450000|     0|     false|
+--------+-----------+------------+----------------+------+----------+
only showing top 5 rows



### D. LOAD — Penyimpanan ke HDFS Parquet Terpartisi
Menyimpan dataset akhir ke direktori Parquet yang terpartisi berdasarkan kolom `kategori` dan memverifikasi integritas pembacaan ulang (5.000 baris utuh).

In [8]:
local_target = "tugas6_hasil_etl_parquet"
username = os.environ.get("USER", "kaky")
hdfs_target = f"hdfs://localhost:9000/user/{username}/tugas6/hasil_etl"

# Simpan ke Parquet terpartisi kategori
df_final.write.mode("overwrite").partitionBy("kategori").parquet(local_target)
print(f"Hasil ETL berhasil disimpan ke direktori lokal: {local_target}")

# Simpan ke HDFS jika cluster aktif
try:
    os.system(f"hdfs dfs -mkdir -p /user/{username}/tugas6 2>/dev/null")
    df_final.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_target)
    print(f"Hasil ETL berhasil disimpan ke HDFS: {hdfs_target}")
except Exception as e:
    print(f"HDFS offline saat runtime ({e.__class__.__name__}). Replikasi lokal terjaga.")

# Verifikasi membaca kembali
df_baca = spark.read.parquet(local_target)
print(f"\nVerifikasi pembacaan ulang: {df_baca.count():,} baris utuh 100%!")
print("Struktur direktori partisi:")
!ls -lh {local_target}

Hasil ETL berhasil disimpan ke direktori lokal: tugas6_hasil_etl_parquet


Hasil ETL berhasil disimpan ke HDFS: hdfs://localhost:9000/user/kaky/tugas6/hasil_etl

Verifikasi pembacaan ulang: 5,000 baris utuh 100%!
Struktur direktori partisi:
Permissions Size User Date Modified Name
drwxr-xr-x     - kaky 30 Sep 23:03   kategori=Elektronik
drwxr-xr-x     - kaky 30 Sep 23:03   kategori=Fashion
drwxr-xr-x     - kaky 30 Sep 23:03   kategori=Kesehatan
drwxr-xr-x     - kaky 30 Sep 23:03   kategori=Makanan
drwxr-xr-x     - kaky 30 Sep 23:03   'kategori=Rumah Tangga'
.rw-r--r--     8 kaky 30 Sep 23:03   ._SUCCESS.crc
.rw-r--r--     0 kaky 30 Sep 23:03  󰡯 _SUCCESS


### E. INSIGHT AKHIR — Analisis Rasio Ulasan per Kategori
Mencari kategori produk dengan persentase transaksi berulasan (`ada_ulasan = True`) paling rendah menggunakan DataFrame API dan Spark SQL.

In [9]:
# Menghitung rasio ulasan per kategori
df_insight = df_final.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_sum(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("transaksi_dengan_ulasan")
).withColumn(
    "persentase_berulasan",
    spark_round((col("transaksi_dengan_ulasan") / col("total_transaksi")) * 100, 2)
).orderBy("persentase_berulasan")

print("=== DISTRIBUSI RASIO ULASAN PER KATEGORI ===")
df_insight.show()

=== DISTRIBUSI RASIO ULASAN PER KATEGORI ===
+------------+---------------+-----------------------+--------------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_berulasan|
+------------+---------------+-----------------------+--------------------+
|     Makanan|            536|                    369|               68.84|
|   Kesehatan|            961|                    662|               68.89|
|     Fashion|           1035|                    726|               70.14|
|Rumah Tangga|            815|                    575|               70.55|
|  Elektronik|           1653|                   1168|               70.66|
+------------+---------------+-----------------------+--------------------+



### Interpretasi Bisnis untuk Tim Pemasaran
Kategori **Makanan (68.84%)** dan **Kesehatan (68.89%)** memiliki persentase transaksi berulasan paling rendah. Temuan ini sangat kritikal bagi kelangsungan bisnis e-commerce karena produk konsumsi dan kesehatan memiliki sensitivitas risiko yang tinggi bagi konsumen baru yang membutuhkan *social proof* (testimoni rasa, higienitas, dan khasiat nyata) sebelum berani membeli. Rendahnya ulasan menandakan pembeli cenderung langsung mengonsumsi produk tanpa terdorong membuka aplikasi kembali.

Rekomendasi strategisnya: tim marketing disarankan meluncurkan program *gamification review* berhadiah cashback koin atau voucher diskon khusus ulasan berfoto, serta mengaktifkan notifikasi pengingat ulasan otomatis yang dikirimkan tepat tiga hari setelah status pesanan terkonfirmasi tiba di tangan konsumen.

### 3. Penutupan Sesi Spark
Menghentikan SparkSession untuk melepaskan alokasi thread CPU dan memori JVM pada sistem host Linux.

In [10]:
spark.stop()
print("SparkSession berhasil dihentikan. Alokasi resource CPU dan RAM telah dikembalikan ke sistem host.")

SparkSession berhasil dihentikan. Alokasi resource CPU dan RAM telah dikembalikan ke sistem host.
